In [1]:
!pip install transformers torch scikit-learn pandas numpy librosa

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 430.4 kB/s eta 0:00:28
   - -------------------------------------- 0.5/12.3 MB 430.4 kB/s eta 0:00:28
   - -------------------------------------- 0.5/12.3 MB 430.4 kB/s eta 0:00:28
   - -------------------------------------- 0.5/12.3 MB 430.4 kB/s eta 0:00:28
   - -------------------------------------- 0.5/12.3 M

In [2]:
import pandas as pd
import numpy as np
from transformers import pipeline
from sklearn.ensemble import IsolationForest
import librosa
import torch
import torch.nn as nn

c:\Users\Sourav Yadav\OneDrive\Desktop\Python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

model_name = "j-hartmann/emotion-english-distilroberta-base"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Downloading weights...")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    device=-1  # Runs on CPU safely
)

def analyze_text_distress(text):
    results = emotion_classifier(text)[0]
    # Filter negative emotions
    distress_scores = {res['label']: res['score'] for res in results if res['label'] in ['fear', 'sadness', 'anger']}
    combined_score = sum(distress_scores.values())
    return combined_score, distress_scores

# Run test
text_input = "I am feeling very isolated and scared about what might happen next."
score, details = analyze_text_distress(text_input)
print("Done!")
print(f"Distress Score: {score:.2f}")
print(f"Details: {details}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 23860.97it/s]

Done!
Distress Score: 0.99
Details: {'sadness': 0.9467265605926514, 'fear': 0.039790160953998566, 'anger': 0.002601517364382744}


In [5]:
import numpy as np
from sklearn.ensemble import IsolationForest

# Creating synthetic behavioral data for the prototype
# Columns: [Screen_Time_Hours, Missed_Checkins, Late_Night_Activity_Hours]
# The first 4 rows are "Normal" behavior, the last row is an "Anomaly" (distress pattern)
behavioral_data = np.array([
    [2.0, 0, 0.5], 
    [1.5, 0, 0.0], 
    [3.0, 1, 1.0], 
    [2.5, 0, 0.5],
    [12.0, 5, 6.0]  # Anomaly: Very high screen time, 5 missed check-ins, 6 hours late night activity
])

print("Training Behavioral Anomaly Model...")
# Train the Isolation Forest
iso_forest = IsolationForest(contamination=0.2, random_state=42)
iso_forest.fit(behavioral_data)

def analyze_behavior(user_metrics):
    # Model returns -1 for an anomaly (distress) and 1 for normal
    prediction = iso_forest.predict([user_metrics])[0]
    return "Anomalous" if prediction == -1 else "Normal"

# Test the model with a suspicious user pattern
sample_behavior = [10.5, 3, 4.5] 
status = analyze_behavior(sample_behavior)

print("Done!")
print(f"Tested metrics: {sample_behavior}")
print(f"Behavior Status: {status}")

Training Behavioral Anomaly Model...
Done!
Tested metrics: [10.5, 3, 4.5]
Behavior Status: Anomalous


In [6]:
import librosa
import numpy as np

def extract_audio_features(file_path):
    try:
        # Attempt to load a real audio file
        y, sr = librosa.load(file_path, duration=3, offset=0.5)
        # Extract MFCC features (industry standard for speech emotion)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        return np.mean(mfcc.T, axis=0)
    except Exception as e:
        # Fallback to synthetic data if no file is found (perfect for rapid prototyping)
        return np.random.rand(40) 

def analyze_audio_emotion(features):
    # Mocking a PyTorch classification output for the prototype
    emotions = ['Neutral', 'Sad', 'Fear', 'Anger']
    # Simulating a prediction favoring distress states
    return np.random.choice(emotions, p=[0.2, 0.3, 0.4, 0.1])

# Test the audio module
test_features = extract_audio_features("dummy_audio.wav")
predicted_emotion = analyze_audio_emotion(test_features)

print("Audio Module Ready!")
print(f"Predicted Voice Emotion: {predicted_emotion}")

Audio Module Ready!
Predicted Voice Emotion: Fear


In [7]:
def dynamic_distress_prediction(text_input, behavioral_metrics, audio_file_path="dummy.wav"):
    total_risk_score = 0
    
    # 1. NLP Weight (Max 40 points)
    nlp_score, _ = analyze_text_distress(text_input)
    total_risk_score += (nlp_score * 40)
    
    # 2. Behavioral Weight (Max 30 points)
    behavior_status = analyze_behavior(behavioral_metrics)
    if behavior_status == "Anomalous":
        total_risk_score += 30
        
    # 3. Audio Weight (Max 30 points)
    audio_features = extract_audio_features(audio_file_path)
    audio_emotion = analyze_audio_emotion(audio_features)
    if audio_emotion in ['Fear', 'Sad', 'Anger']:
        total_risk_score += 30
        
    # Determine Risk Category based on your SIH flowchart
    if total_risk_score >= 80:
        category = "Critical"
    elif total_risk_score >= 60:
        category = "High Risk"
    elif total_risk_score >= 35:
        category = "Moderate Risk"
    else:
        category = "Low Risk"
        
    return {
        "Total_Score": round(total_risk_score, 2),
        "Risk_Category": category,
        "Breakdown": {
            "NLP_Distress": round(nlp_score, 2),
            "Behavior": behavior_status,
            "Audio_Emotion": audio_emotion
        }
    }

# Execute a full pipeline simulation (a simulated victim check-in)
test_text = "I haven't been sleeping well and I feel extremely nervous about leaving my house."
test_behavior = [10.5, 3, 4.5] # Suspicious behavioral metrics

print("--- BytePulse Multimodal Engine Output ---")
final_prediction = dynamic_distress_prediction(test_text, test_behavior)
print(f"Dynamic Score: {final_prediction['Total_Score']}/100")
print(f"Risk Category: {final_prediction['Risk_Category']}")
print(f"Details: {final_prediction['Breakdown']}")

--- BytePulse Multimodal Engine Output ---
Dynamic Score: 99.86/100
Risk Category: Critical
Details: {'NLP_Distress': 1.0, 'Behavior': 'Anomalous', 'Audio_Emotion': np.str_('Sad')}


In [8]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib
import os

# 1. Generate 5,000 rows of synthetic behavioral data
print("Generating synthetic dataset...")
np.random.seed(42)

# Normal behavior (95% of users)
normal_data = np.column_stack([
    np.random.uniform(1.0, 4.0, 4750), # Screen time (1-4 hrs)
    np.zeros(4750),                    # Missed check-ins (0)
    np.random.uniform(0.0, 1.0, 4750)  # Late night activity (0-1 hrs)
])

# Anomalous/Distress behavior (5% of users)
anomalous_data = np.column_stack([
    np.random.uniform(8.0, 14.0, 250), # Screen time (8-14 hrs)
    np.random.randint(2, 6, 250),      # Missed check-ins (2-5)
    np.random.uniform(4.0, 8.0, 250)   # Late night activity (4-8 hrs)
])

# Combine into a single DataFrame and shuffle
full_dataset = np.vstack([normal_data, anomalous_data])
np.random.shuffle(full_dataset)
df = pd.DataFrame(full_dataset, columns=['Screen_Time_Hours', 'Missed_Checkins', 'Late_Night_Hours'])

# Save the dataset to the data folder you created earlier
df.to_csv('./data/synthetic_behavioral_data.csv', index=False)
print("Saved dataset to ./data/synthetic_behavioral_data.csv")

# 2. Train the Machine Learning Model
print("Training the Isolation Forest model...")
# contamination=0.05 because we deliberately made 5% of our data anomalous
real_iso_forest = IsolationForest(contamination=0.05, random_state=42)
real_iso_forest.fit(df.values)

# 3. Export the trained model to the models folder
joblib.dump(real_iso_forest, './models/behavioral_anomaly_model.pkl')
print("Successfully trained and saved model to ./models/behavioral_anomaly_model.pkl")

# 4. Quick Test
test_user = [11.5, 4, 6.0]
prediction = real_iso_forest.predict([test_user])[0]
status = "Anomalous" if prediction == -1 else "Normal"
print(f"Test prediction for metrics {test_user}: {status}")

Generating synthetic dataset...
Saved dataset to ./data/synthetic_behavioral_data.csv
Training the Isolation Forest model...
Successfully trained and saved model to ./models/behavioral_anomaly_model.pkl
Test prediction for metrics [11.5, 4, 6.0]: Anomalous


In [10]:
!pip install fastapi uvicorn joblib pydantic --default-timeout=1000

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 560.1 kB/s eta 0:00:03
   ---------- ----------------------------- 0.5/2.0 MB 560.1 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 524.3 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 524.3 kB/s eta 0:00:03
   -------------------- ------------------- 1.0/2.0 MB 565.8 kB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 565.8 kB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 565.8 kB/s eta 0:00:02
   ------------------------- -------------- 1.3/2.0 MB 528.5 kB/s eta 0:00:02
   ------------------------- --